In [1]:

from pathlib import Path
import pandas as pd

# Locate the project folder.
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

# Read the copied files as text.
# Keeping original values helps us detect invalid data during validation.
input_files = {
    "KAUST": "KAUST_2024_2025_cleaned.csv",
    "KFUPM": "KFUPM_cleaned.csv",
}

datasets_to_validate = {}

for source, filename in input_files.items():
    file_path = project_dir / "data" / "interim" / filename

    df = pd.read_csv(
        file_path,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
    )

    datasets_to_validate[source] = df

    print(f"\n{source}")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("University values:", df["university"].unique().tolist())
    print("Year values:", df["publication_year"].unique().tolist())


KAUST
Rows: 114
Columns: 13
University values: ['KAUST']
Year values: ['2024', '2025']

KFUPM
Rows: 48
Columns: 13
University values: ['KFUPM']
Year values: ['2023', '2024', '2025', '2026']


In [2]:

# Expected columns from the team's README.
# FINAL TEAM SCHEMA was finalized to
# All 13 columns must exist, but only 7 require a value.

expected_columns = [
    "research_id",
    "university",
    "title",
    "authors",
    "publication_year",
    "publication_date",
    "abstract",
    "research_field",
    "tech_category",
    "journal",
    "doi",
    "url",
    "source",
]

# Reject records where any of these fields is missing.
required_fields = [
    "research_id",
    "university",
    "title",
    "publication_year",
    "source",
    "doi",
    "url",
    "authors",
]

optional_fields = [
    "abstract",
    "publication_date",
    "research_field",
    "tech_category",
    "journal",
]
# Missing values here are allowed.
# Missing DOIs will be flagged for later enrichment.




# These placeholders count as missing, ignoring case and whitespace.
missing_markers = {"", "null", "none", "nan", "n/a"}

# Agreed publication-year range, including 2026.
min_year = 2023
max_year = 2026

expected_universities = {
    "KAUST": "KAUST",
    "KFUPM": "KFUPM",
    "KSU": "KSU",
}

# Verify that every column belongs to exactly one category.
assert set(required_fields).isdisjoint(optional_fields)
assert set(required_fields) | set(optional_fields) == set(expected_columns)

print("Schema defined: 8 mandatory and 5 optional fields.")

Schema defined: 8 mandatory and 5 optional fields.


In [3]:


import re
from datetime import datetime
from urllib.parse import urlsplit


def validate_dataset(df, expected_university, duplicate_ids=None):
    """Validate a copy; preserve original input values."""

    # All 13 columns must exist, including optional columns.
    missing_columns = sorted(set(expected_columns) - set(df.columns))
    extra_columns = sorted(set(df.columns) - set(expected_columns))

    if missing_columns or extra_columns:
        raise ValueError(
            f"Schema mismatch. Missing columns: {missing_columns}; "
            f"extra: {extra_columns}"
        )

    # Recognize blanks and agreed missing-value placeholders.
    def clean_value(value):
        if pd.isna(value):
            return ""
        text = str(value).strip()
        return "" if text.casefold() in missing_markers else text

    checked = df.copy()

        # Store missing optional values as blanks in the working copy.
    for field in optional_fields:
        missing = checked[field].map(clean_value).eq("")
        checked.loc[missing, field] = ""

    # Detect duplicate IDs within this file and across supplied files.
    duplicates = set() if duplicate_ids is None else set(duplicate_ids)
    ids = checked["research_id"].map(clean_value)
    duplicates.update(ids[ids.ne("") & ids.duplicated(keep=False)])

    failure_reasons = []

    for _, row in checked.iterrows():
        values = {
            column: clean_value(row[column])
            for column in expected_columns
        }
        errors = []

        # Enforce the eight mandatory fields.
        for field in required_fields:
            if not values[field]:
                errors.append(f"{field}: required value missing")

        research_id = values["research_id"]
        if research_id and research_id in duplicates:
            errors.append("research_id: duplicate across input records")

        university = values["university"]
        if university and university != expected_university:
            errors.append("university: unexpected value")

        year = values["publication_year"]
        if year:
            if not re.fullmatch(r"[0-9]{4}", year):
                errors.append("publication_year: expected four-digit integer")
            elif not min_year <= int(year) <= max_year:
                errors.append(
                    f"publication_year: outside {min_year}-{max_year}"
                )

        # An optional date may be missing; a supplied date must be valid.
        date = values["publication_date"]
        if date:
            try:
                if not re.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}", date):
                    raise ValueError
                datetime.strptime(date, "%Y-%m-%d")
            except ValueError:
                errors.append("publication_date: invalid YYYY-MM-DD date")

        # Missing DOI was already flagged above.
        # This checks populated DOI syntax, not online resolution.
        doi = values["doi"]
        if doi and not re.fullmatch(r"10\.[0-9]{4,9}/\S+", doi):
            errors.append("doi: invalid normalized DOI format")

        url = values["url"]
        if url:
            try:
                parts = urlsplit(url)
                valid_url = (
                    parts.scheme.lower() in {"http", "https"}
                    and bool(parts.hostname)
                    and not any(character.isspace() for character in url)
                )
            except ValueError:
                valid_url = False

            if not valid_url:
                errors.append("url: invalid HTTP(S) URL")

        # Record every failed rule for the row.
        failure_reasons.append("; ".join(errors))

    checked["failure_reason"] = pd.Series(
        failure_reasons, index=checked.index, dtype="string"
    )

    accepted = checked.loc[checked["failure_reason"].eq("")].copy()
    rejected = checked.loc[checked["failure_reason"].ne("")].copy()

    assert len(accepted) + len(rejected) == len(df)

    return {"accepted": accepted, "rejected": rejected}



In [4]:

# Artificial records used only for testing.
base_record = {column: "" for column in expected_columns}
base_record.update({
    "research_id": "TEST_ONLY_001",
    "university": "KAUST",
    "title": "TEST ONLY - validation example",
    "authors": "Test Author",
    "publication_year": "2024",
    "doi": "10.1234/test-example",
    "url": "https://example.org/test-only",
    "source": "TEST_FIXTURE",
})

test_results = []


def record_pass(name):
    test_results.append({"test": name, "status": "passed"})
    print("PASS:", name)


def check_case(name, changes, expected_failed_fields):
    record = base_record.copy()
    record.update(changes)

    result = validate_dataset(pd.DataFrame([record]), "KAUST")

    if expected_failed_fields:
        assert len(result["rejected"]) == 1, f"{name}: should reject"

        reason = result["rejected"].iloc[0]["failure_reason"]
        actual_fields = {
            message.split(":", 1)[0]
            for message in reason.split("; ")
        }

        assert actual_fields == set(expected_failed_fields), (
            f"{name}: unexpected failure reasons: {reason}"
        )
    else:
        assert len(result["accepted"]) == 1, f"{name}: should accept"

    record_pass(name)


# All optional fields may be blank or contain missing placeholders.
check_case("Optional fields may be blank", {}, [])

check_case(
    "Optional null placeholders accepted",
    {field: "null" for field in optional_fields},
    [],
)



# Every mandatory field must reject blanks and missing placeholders.
for field in required_fields:
    for label, value in [
        ("blank", "   "),
        ("null placeholder", "null"),
        ("missing value", None),
    ]:
        check_case(
            f"{field}: {label} rejected",
            {field: value},
            [field],
        )

# Year boundaries and formats.
check_case("Year 2023 accepted", {"publication_year": "2023"}, [])
check_case("Year 2026 accepted", {"publication_year": "2026"}, [])
check_case("Year 2022 rejected", {"publication_year": "2022"}, ["publication_year"])
check_case("Year 2027 rejected", {"publication_year": "2027"}, ["publication_year"])
check_case(
    "Malformed year rejected",
    {"publication_year": "2024.5"},
    ["publication_year"],
)

# Optional dates must be valid when populated.
check_case("Leap day accepted", {"publication_date": "2024-02-29"}, [])
check_case(
    "Impossible date rejected",
    {"publication_date": "2024-02-30"},
    ["publication_date"],
)
check_case(
    "Partial date rejected",
    {"publication_date": "2024-02"},
    ["publication_date"],
)

check_case("Malformed DOI rejected", {"doi": "invalid"}, ["doi"])
check_case("Invalid URL rejected", {"url": "not-a-url"}, ["url"])
check_case(
    "Unexpected university rejected",
    {"university": "UNKNOWN"},
    ["university"],
)
check_case(
    "All failed fields reported",
    {"title": "", "authors": "", "doi": "invalid"},
    ["title", "authors", "doi"],
)

# Duplicate IDs within a file.
duplicate_input = pd.DataFrame([base_record.copy(), base_record.copy()])
result = validate_dataset(duplicate_input, "KAUST")
assert len(result["rejected"]) == 2
assert result["rejected"]["failure_reason"].str.contains(
    "research_id: duplicate", regex=False
).all()
record_pass("Duplicate IDs rejected")

# Duplicate IDs across files.
result = validate_dataset(
    pd.DataFrame([base_record]),
    "KAUST",
    duplicate_ids={"TEST_ONLY_001"},
)
assert len(result["rejected"]) == 1
assert "research_id: duplicate" in result["rejected"].iloc[0]["failure_reason"]
record_pass("Cross-file duplicate rejected")

# All schema columns must exist, even optional columns.
for missing_column in ["title", "abstract"]:
    try:
        validate_dataset(
            pd.DataFrame([base_record]).drop(columns=[missing_column]),
            "KAUST",
        )
    except ValueError as error:
        assert f"Missing columns: ['{missing_column}']" in str(error)
    else:
        raise AssertionError(f"Missing column not detected: {missing_column}")

    record_pass(f"Missing {missing_column} column detected")

# Empty input must preserve the output structure.
result = validate_dataset(
    pd.DataFrame(columns=expected_columns),
    "KAUST",
)
assert result["accepted"].empty
assert result["rejected"].empty
assert "failure_reason" in result["rejected"].columns
record_pass("Empty input handled")

print(f"\nAll {len(test_results)} tests passed.")
print("Test records were not added to the research datasets.")


PASS: Optional fields may be blank
PASS: Optional null placeholders accepted
PASS: research_id: blank rejected
PASS: research_id: null placeholder rejected
PASS: research_id: missing value rejected
PASS: university: blank rejected
PASS: university: null placeholder rejected
PASS: university: missing value rejected
PASS: title: blank rejected
PASS: title: null placeholder rejected
PASS: title: missing value rejected
PASS: publication_year: blank rejected
PASS: publication_year: null placeholder rejected
PASS: publication_year: missing value rejected
PASS: source: blank rejected
PASS: source: null placeholder rejected
PASS: source: missing value rejected
PASS: doi: blank rejected
PASS: doi: null placeholder rejected
PASS: doi: missing value rejected
PASS: url: blank rejected
PASS: url: null placeholder rejected
PASS: url: missing value rejected
PASS: authors: blank rejected
PASS: authors: null placeholder rejected
PASS: authors: missing value rejected
PASS: Year 2023 accepted
PASS: Year 

In [5]:
import json

# Inspect each saved KSU candidate file without modifying it.
for year in [2023, 2024, 2025]:
    path = (
        project_dir / "data" / "interim"
        / f"ksu_tech_candidates_{year}.json"
    )

    records = json.loads(path.read_text(encoding="utf-8"))

    print(f"\nKSU {year}: {len(records):,} records")

    if records:
        available_fields = set(records[0].keys())

        print("First record's fields:", list(records[0].keys()))
        print(
            "README columns missing from the first record:",
            sorted(set(expected_columns) - available_fields),
        )


KSU 2023: 940 records
First record's fields: ['Authors', 'Article Title', 'Source Title', 'Document Type', 'Author Keywords', 'Abstract', 'Affiliations', 'DOI', 'source_file_year', 'source_row_index', 'tech_matched_terms']
README columns missing from the first record: ['abstract', 'authors', 'doi', 'journal', 'publication_date', 'publication_year', 'research_field', 'research_id', 'source', 'tech_category', 'title', 'university', 'url']

KSU 2024: 1,306 records
First record's fields: ['Authors', 'Article Title', 'Source Title', 'Document Type', 'Author Keywords', 'Affiliations', 'source_file_year', 'source_row_index', 'tech_matched_terms']
README columns missing from the first record: ['abstract', 'authors', 'doi', 'journal', 'publication_date', 'publication_year', 'research_field', 'research_id', 'source', 'tech_category', 'title', 'university', 'url']

KSU 2025: 1,289 records
First record's fields: ['Authors', 'Article Title', 'Source Title', 'Document Type', 'Author Keywords', 'Abs

In [6]:

# Build standardized KSU records without changing the source files.
ksu_rows = []
ksu_provenance = []

for year in [2023, 2024, 2025]:
    input_path = (
        project_dir / "data" / "interim"
        / f"ksu_tech_candidates_{year}.json"
    )
    records = json.loads(input_path.read_text(encoding="utf-8"))

    source_url = (
        "https://"
        "data.ksu.edu.sa/sites/data.ksu.edu.sa/files/users/user976/"
        f"KSU-DMO-OD-DATASET-SciResarch-Publications-{year}.json"
    )

    for record in records:
        # Confirm the record belongs to this annual file.
        assert int(record["source_file_year"]) == year

        original_index = record["source_row_index"]
        research_id = f"KSU_{year}_{original_index}"

        # Convert available scalar values to trimmed text.
        def get_text(field):
            value = record.get(field)
            return "" if value is None else str(value).strip()

        # Normalize DOI prefixes while preserving the original in provenance.
        original_doi = get_text("DOI")
        normalized_doi = re.sub(
            r"^(?:https?://(?:dx\.)?doi\.org/|doi:\s*)",
            "",
            original_doi,
            flags=re.IGNORECASE,
        ).strip().lower()

        ksu_rows.append({
            "research_id": research_id,
            "university": "KSU",
            "title": get_text("Article Title"),
            "authors": get_text("Authors"),

            # Agreed rule: use KSU's annual file year.
            "publication_year": str(year),

            # Do not invent a complete date.
            "publication_date": "",
            "abstract": get_text("Abstract"),

            # Keywords and matched terms are not verified classifications.
            "research_field": "",
            "tech_category": "",
            "journal": get_text("Source Title"),
            "doi": normalized_doi,

            # This is a dataset source link, not an individual paper URL.
            "url": source_url,
            "source": "KSU",
        })

        # Preserve evidence outside the fixed 13-column schema.
        ksu_provenance.append({
            "research_id": research_id,
            "source_file_year": year,
            "source_row_index": original_index,
            "input_file": input_path.relative_to(project_dir).as_posix(),
            "source_url": source_url,
            "publication_year_basis": "KSU annual dataset year",
            "url_basis": "Annual dataset download URL",
            "original_doi": original_doi,
            "affiliations": record.get("Affiliations"),
            "author_keywords": record.get("Author Keywords"),
            "document_type": record.get("Document Type"),
            "tech_matched_terms": record.get("tech_matched_terms", []),
        })

ksu_standardized = pd.DataFrame(ksu_rows, columns=expected_columns)

assert ksu_standardized["research_id"].is_unique
assert len(ksu_standardized) == len(ksu_provenance)

print("Standardized KSU records:", len(ksu_standardized))
print("Columns:", len(ksu_standardized.columns))
print("\nRecords by reported year:")
print(ksu_standardized.groupby("publication_year").size())
print("\nCreated in memory only. Original files unchanged.")


Standardized KSU records: 3535
Columns: 13

Records by reported year:
publication_year
2023     940
2024    1306
2025    1289
dtype: int64

Created in memory only. Original files unchanged.


In [7]:

# Collect the prepared datasets without changing their original files.
all_validation_inputs = {
    **datasets_to_validate,
    "KSU": ksu_standardized,
}

university_labels = {
    "KAUST": "KAUST",
    "KFUPM": "KFUPM",
    "KSU": "KSU",
}

# Detect duplicate IDs across all three sources.
combined_ids = pd.concat(
    [
        df["research_id"].fillna("").astype(str).str.strip()
        for df in all_validation_inputs.values()
    ],
    ignore_index=True,
)

valid_id_mask = ~combined_ids.str.casefold().isin(missing_markers)

all_duplicate_ids = set(
    combined_ids[
        valid_id_mask
        & combined_ids.duplicated(keep=False)
    ]
)

# Apply the final validator with the eight mandatory fields.
all_validation_results = {}
validation_summary = []

for source, df in all_validation_inputs.items():
    result = validate_dataset(
        df,
        university_labels[source],
        duplicate_ids=all_duplicate_ids,
    )

    all_validation_results[source] = result

    input_count = len(df)
    accepted_count = len(result["accepted"])
    rejected_count = len(result["rejected"])

    assert accepted_count + rejected_count == input_count

    validation_summary.append({
        "source": source,
        "input_rows": input_count,
        "accepted_rows": accepted_count,
        "rejected_rows": rejected_count,
        "min_year": min_year,
        "max_year": max_year,
    })

    print(f"\n{source}")
    print("Input:", input_count)
    print("Accepted:", accepted_count)
    print("Rejected:", rejected_count)

    for _, row in result["rejected"].head(5).iterrows():
        print("-", row["research_id"], "|", row["failure_reason"])

print("\nValidation complete in memory. Original files unchanged.")


KAUST
Input: 114
Accepted: 114
Rejected: 0

KFUPM
Input: 48
Accepted: 0
Rejected: 48
- KFUPM-142340 | doi: required value missing
- KFUPM-142314 | doi: required value missing
- KFUPM-143171 | doi: required value missing
- KFUPM-143135 | doi: required value missing
- KFUPM-143136 | doi: required value missing

KSU
Input: 3535
Accepted: 1473
Rejected: 2062
- KSU_2023_992 | doi: required value missing
- KSU_2023_2010 | doi: required value missing
- KSU_2023_2024 | doi: required value missing
- KSU_2023_2026 | doi: required value missing
- KSU_2023_2106 | doi: required value missing

Validation complete in memory. Original files unchanged.


end of review


In [8]:
expected_test_count = 19 + 3 * len(required_fields)


assert len(test_results) == expected_test_count
assert all(test["status"] == "passed" for test in test_results)

print(f"All {len(test_results)} current-schema tests passed.")

All 43 current-schema tests passed.


In [9]:
from datetime import timezone
from collections import Counter

# Create a new output folder without replacing earlier results.
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
final_output_dir = (
    project_dir / "data" / "interim"
    / "rana_saad_validation" / f"mandatory_doi_{run_stamp}"
)
final_output_dir.mkdir(parents=True, exist_ok=False)

accepted_parts = []
rejected_parts = []
rule_counts = []

for source, result in all_validation_results.items():
    accepted = result["accepted"].drop(columns=["failure_reason"]).copy()
    rejected = result["rejected"].copy()

    # Confirm no artificial test records entered the real outputs.
    for frame in [accepted, rejected]:
        assert not frame["source"].eq("TEST_FIXTURE").any()

    accepted.to_csv(
        final_output_dir / f"{source}_validated.csv",
        index=False, encoding="utf-8",
    )
    rejected.to_csv(
        final_output_dir / f"{source}_rejected.csv",
        index=False, encoding="utf-8",
    )

    accepted_parts.append(accepted)
    rejected_parts.append(rejected)

    # Count each failed rule separately.
    counts = Counter(
        reason
        for reasons in rejected["failure_reason"]
        for reason in reasons.split("; ")
        if reason
    )

    for reason, count in counts.items():
        rule_counts.append({
            "source": source,
            "failure_reason": reason,
            "failed_rows": count,
        })

# Combine validation outputs—not the final transformed dataset.
accepted_all = pd.concat(accepted_parts, ignore_index=True)
rejected_all = pd.concat(rejected_parts, ignore_index=True)

assert len(accepted_all) + len(rejected_all) == sum(
    len(df) for df in all_validation_inputs.values()
)
assert accepted_all["research_id"].str.strip().is_unique

# Save and reopen combined files to verify their contents.
for filename, frame in [
    ("validated.csv", accepted_all),
    ("rejected.csv", rejected_all),
]:
    path = final_output_dir / filename
    frame.to_csv(path, index=False, encoding="utf-8")

    reopened = pd.read_csv(
        path, dtype=str, keep_default_na=False, encoding="utf-8"
    )
    pd.testing.assert_frame_equal(
        reopened, frame.reset_index(drop=True), check_dtype=False
    )

pd.DataFrame(validation_summary).to_csv(
    final_output_dir / "validation_summary.csv", index=False
)

pd.DataFrame(test_results).to_csv(
    final_output_dir / "validation_test_results.csv", index=False
)

pd.DataFrame(
    rule_counts,
    columns=["source", "failure_reason", "failed_rows"],
).to_csv(
    final_output_dir / "validation_failures_by_rule.csv", index=False
)

# Preserve KSU mapping decisions and original source references.
(final_output_dir / "ksu_provenance.json").write_text(
    json.dumps(ksu_provenance, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# Record the exact rules used for this run.
run_rules = {
    "required_fields": required_fields,
    "optional_fields": optional_fields,
    "missing_markers": sorted(missing_markers),
    "publication_year_range": [min_year, max_year],
    "ksu_year_basis": "KSU annual dataset year",
    "doi_check": "Syntax check; does not prove DOI resolution",
    "enrichment_status": "Previously reviewed matches not yet applied",
    "sources_validated": list(all_validation_inputs),
    "tests_passed": len(test_results),
}

(final_output_dir / "validation_rules.json").write_text(
    json.dumps(run_rules, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Accepted records saved:", len(accepted_all))
print("Rejected records preserved:", len(rejected_all))
print("Tests passed:", len(test_results))
print("Output folder:", final_output_dir)
print("Original inputs unchanged.")

Accepted records saved: 1587
Rejected records preserved: 2110
Tests passed: 43
Output folder: c:\Users\ausu\Desktop\saudi-tech-research\data\interim\rana_saad_validation\mandatory_doi_20260913T103258216406Z
Original inputs unchanged.


In [10]:
# Load the five-paper trial summary.
interim_dir = project_dir / "data" / "interim"

trial_summary = json.loads(
    (interim_dir / "ksu_five_paper_trial_summary.json")
    .read_text(encoding="utf-8")
)

# Select only the four reviewed matches.
reviewed_matches = [
    item for item in trial_summary
    if item["match_status"] == "reviewed_match"
]

assert len(reviewed_matches) == 4, "Expected four reviewed matches."

# Check each match against the current standardized dataset.
for match in reviewed_matches:
    research_id = (
        f"KSU_{match['source_file_year']}_{match['source_row_index']}"
    )

    rows = ksu_standardized.loc[
        ksu_standardized["research_id"].eq(research_id)
    ]

    assert len(rows) == 1, f"Missing or duplicate record: {research_id}"
    row = rows.iloc[0]

    # IDs and titles must agree before we attach metadata.
    assert row["title"].strip() == match["title"].strip(), (
        f"Title mismatch: {research_id}"
    )

    doi = match["confirmed_doi"].strip().lower()
    assert re.fullmatch(r"10\.[0-9]{4,9}/\S+", doi), (
        f"Invalid saved DOI: {research_id}"
    )

    print("\nID:", research_id)
    print("Title:", row["title"])
    print("Current DOI:", repr(row["doi"]))
    print("Reviewed DOI:", doi)
    print("Evidence file:", match["evidence_file"])

print("\nChecks complete. No metadata has been changed yet.")


ID: KSU_2024_69
Title: 3-D Trajectory Optimization and Communication Resources Allocation in UAV-Assisted IoT Networks for Sustainable Industry 5.0
Current DOI: ''
Reviewed DOI: 10.1109/tce.2023.3325131
Evidence file: ksu_sample_enrichment.json

ID: KSU_2023_2010
Title: Empowering smart cities: High-altitude platforms based Mobile Edge Computing and Wireless Power Transfer for efficient IoT data processing
Current DOI: ''
Reviewed DOI: 10.1016/j.iot.2023.100986
Evidence file: ksu_four_paper_review_decisions.json

ID: KSU_2024_85
Title: 6G-Enabled Consumer Electronics Device Intrusion Detection With Federated Meta-Learning and Digital Twins in a Meta-Verse Environment
Current DOI: ''
Reviewed DOI: 10.1109/tce.2023.3321846
Evidence file: ksu_paper3_openalex_review.json

ID: KSU_2024_86
Title: 6GTelMED: Resources Recommendation Framework on 6G-Enabled Distributed Telemedicine Using Edge-AI
Current DOI: ''
Reviewed DOI: 10.1109/tce.2024.3473291
Evidence file: ksu_paper4_openalex_review.js

In [11]:
# Once these checks pass, we’ll fill the missing DOIs and available abstracts, 
# preserve their evidence, and rerun validation.

#All four matches align with your KSU records. 
# Now we can fill their missing DOIs and the 
# three available abstracts in a separate working copy


# Keep the pre-enrichment dataset unchanged.
ksu_enriched = ksu_standardized.copy()
enrichment_log = []

def is_missing(value):
    return (
        pd.isna(value)
        or str(value).strip().casefold() in missing_markers
    )

for match in reviewed_matches:
    research_id = (
        f"KSU_{match['source_file_year']}_{match['source_row_index']}"
    )
    row_index = ksu_enriched.index[
        ksu_enriched["research_id"].eq(research_id)
    ][0]

    evidence_path = interim_dir / match["evidence_file"]
    evidence = json.loads(evidence_path.read_text(encoding="utf-8"))

    # One evidence file contains multiple review decisions.
    if isinstance(evidence, list):
        evidence = next(
            item for item in evidence
            if item["source_record"]["source_file_year"]
            == match["source_file_year"]
            and item["source_record"]["source_row_index"]
            == match["source_row_index"]
        )

    doi = match["confirmed_doi"].strip().lower()
    assert evidence["confirmed_doi"].strip().lower() == doi

    changes = []

    # Fill a missing DOI; never overwrite a conflicting existing DOI.
    current_doi = ksu_enriched.at[row_index, "doi"]
    if is_missing(current_doi):
        ksu_enriched.at[row_index, "doi"] = doi
        changes.append("doi")
    else:
        assert str(current_doi).strip().lower() == doi

    # Fill the abstract only when the original is missing.
    abstract = evidence.get("abstract")
    if (
        is_missing(ksu_enriched.at[row_index, "abstract"])
        and not is_missing(abstract)
    ):
        ksu_enriched.at[row_index, "abstract"] = abstract
        changes.append("abstract")

    enrichment_log.append({
        "research_id": research_id,
        "fields_added": changes,
        "confirmed_doi": doi,
        "doi_source": "Crossref",
        "abstract_source": "OpenAlex" if "abstract" in changes else None,
        "evidence_file": match["evidence_file"],
        "quality_flags": evidence.get("quality_flags", []),
        "year_rule": "Retained KSU annual dataset year",
    })

# Validate the enriched copy with the same rules.
ksu_enriched_result = validate_dataset(
    ksu_enriched,
    "KSU",
    duplicate_ids=all_duplicate_ids,
)

print("DOIs added:", sum("doi" in x["fields_added"] for x in enrichment_log))
print(
    "Abstracts added:",
    sum("abstract" in x["fields_added"] for x in enrichment_log),
)
print("KSU accepted:", len(ksu_enriched_result["accepted"]))
print("KSU rejected:", len(ksu_enriched_result["rejected"]))
print("Original inputs and baseline outputs unchanged.")

DOIs added: 4
Abstracts added: 3
KSU accepted: 1477
KSU rejected: 2058
Original inputs and baseline outputs unchanged.


In [12]:
# Create a separate output folder for the enriched validation run.
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
enriched_output_dir = (
    project_dir / "data" / "interim"
    / "rana_saad_validation" / f"enriched_{stamp}"
)
enriched_output_dir.mkdir(parents=True, exist_ok=False)

# Use the new KSU results and the unchanged KAUST/KFUPM results.
enriched_results = {
    **all_validation_results,
    "KSU": ksu_enriched_result,
}

accepted_parts = []
rejected_parts = []
summary_rows = []
failure_rows = []

for source, result in enriched_results.items():
    accepted = result["accepted"].drop(columns=["failure_reason"]).copy()
    rejected = result["rejected"].copy()

    assert len(accepted) + len(rejected) == len(all_validation_inputs[source])

    for frame in [accepted, rejected]:
        assert not frame["source"].eq("TEST_FIXTURE").any()

    accepted_parts.append(accepted)
    rejected_parts.append(rejected)

    summary_rows.append({
        "source": source,
        "input_rows": len(all_validation_inputs[source]),
        "accepted_rows": len(accepted),
        "rejected_rows": len(rejected),
    })

    counts = Counter(
        reason
        for reasons in rejected["failure_reason"]
        for reason in reasons.split("; ")
        if reason
    )
    for reason, count in counts.items():
        failure_rows.append({
            "source": source,
            "failure_reason": reason,
            "failed_rows": count,
        })

    accepted.to_csv(
        enriched_output_dir / f"{source}_validated.csv",
        index=False, encoding="utf-8",
    )
    rejected.to_csv(
        enriched_output_dir / f"{source}_rejected.csv",
        index=False, encoding="utf-8",
    )

# Combine the accepted and rejected records.
accepted_combined = pd.concat(accepted_parts, ignore_index=True)
rejected_combined = pd.concat(rejected_parts, ignore_index=True)

assert len(accepted_combined) + len(rejected_combined) == 3697
assert accepted_combined["research_id"].str.strip().is_unique

# Save and verify the combined CSVs.
for filename, frame in [
    ("validated.csv", accepted_combined),
    ("rejected.csv", rejected_combined),
]:
    path = enriched_output_dir / filename
    frame.to_csv(path, index=False, encoding="utf-8")

    reopened = pd.read_csv(
        path, dtype=str, keep_default_na=False, encoding="utf-8"
    )
    pd.testing.assert_frame_equal(
        reopened, frame.reset_index(drop=True), check_dtype=False
    )

# Save counts, test results and rejection reasons.
pd.DataFrame(summary_rows).to_csv(
    enriched_output_dir / "validation_summary.csv", index=False
)
pd.DataFrame(test_results).to_csv(
    enriched_output_dir / "validation_test_results.csv", index=False
)
pd.DataFrame(
    failure_rows,
    columns=["source", "failure_reason", "failed_rows"],
).to_csv(
    enriched_output_dir / "validation_failures_by_rule.csv", index=False
)

# Preserve the rules and links to the original enrichment evidence.
enriched_rules = {
    **run_rules,
    "enrichment_status": "Applied 4 reviewed DOIs and 3 abstracts",
    "baseline_output_folder": final_output_dir.relative_to(
        project_dir
    ).as_posix(),
}

for filename, content in [
    ("enrichment_log.json", enrichment_log),
    ("ksu_provenance.json", ksu_provenance),
    ("validation_rules.json", enriched_rules),
]:
    (enriched_output_dir / filename).write_text(
        json.dumps(content, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

print("Accepted:", len(accepted_combined))
print("Rejected and preserved:", len(rejected_combined))
print("Saved and verified:", enriched_output_dir)
print("Baseline outputs and original inputs unchanged.")

Accepted: 1591
Rejected and preserved: 2106
Saved and verified: c:\Users\ausu\Desktop\saudi-tech-research\data\interim\rana_saad_validation\enriched_20260913T103301680594Z
Baseline outputs and original inputs unchanged.


Inspect files before adding them to validation.

In [13]:
from pathlib import Path
import pandas as pd

# Locate the project folder.
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

# Read local copies without modifying either input file.
new_input_files = {
    "KAUST_2023": "KAUST_2023_cleaned.csv",
    "PNU": "PNU_cleaned.csv",
}

new_datasets = {}

for label, filename in new_input_files.items():
    path = project_dir / "data" / "interim" / filename

    df = pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
    )
    new_datasets[label] = df

    print(f"\n--- {label} ---")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    for field in ["university", "publication_year"]:
        if field in df.columns:
            print(f"{field} values:", df[field].unique().tolist())

    if "publication_date" in df.columns:
        dates = df["publication_date"].str.strip()
        print("Sample populated dates:", dates[dates.ne("")].head(3).tolist())

    print("\nFirst record:")
    print(df.head(1).to_dict(orient="records"))

print("\nInspection only. Input files unchanged.")


--- KAUST_2023 ---
Rows: 928
Columns: ['research_id', 'university', 'title', 'authors', 'publication_year', 'publication_date', 'abstract', 'research_field', 'tech_category', 'journal', 'doi', 'url', 'source']
university values: ['KAUST']
publication_year values: ['2023']
Sample populated dates: ['2023-04-11 00:00:00+00:00', '2023-04-11 00:00:00+00:00', '2023-03-10 00:00:00+00:00']

First record:
[{'research_id': 'http://hdl.handle.net/10754/691073', 'university': 'KAUST', 'title': 'Elucidating the Role of Contact-Induced Gap States and Passivation Molecules at Perovskite/Metal Contacts', 'authors': 'Pradhan, Rakesh R.; Eswaran, Mathan Kumar; Subbiah, Anand Selvin; Babayigit, Aslihan; De Wolf, Stefaan; Schwingenschlögl, Udo', 'publication_year': '2023', 'publication_date': '2023-04-11 00:00:00+00:00', 'abstract': 'Metal halide perovskite solar cells hold great promise as an efficient and cost-effective photovoltaic technology. However, carrier recombination at their contacts impedes p

In [14]:
# Final mandatory fields agreed by the team.
mandatory_fields = [
    "research_id", "university", "title", "publication_year",
    "source", "doi", "url", "authors",
]

missing_values = {"", "null", "none", "nan", "n/a"}

for label, df in new_datasets.items():
    print(f"\n--- {label}: {len(df):,} records ---")

    any_required_missing = pd.Series(False, index=df.index)

    for field in mandatory_fields:
        values = (
            df[field].fillna("").astype(str).str.strip().str.casefold()
        )
        missing = values.isin(missing_values)
        any_required_missing |= missing

        print(f"{field}: {missing.sum():,} missing")

    print(
        "Records missing at least one mandatory field:",
        f"{any_required_missing.sum():,}",
    )

print("\nAudit only. No files changed.")


--- KAUST_2023: 928 records ---
research_id: 0 missing
university: 0 missing
title: 0 missing
publication_year: 0 missing
source: 0 missing
doi: 113 missing
url: 0 missing
authors: 1 missing
Records missing at least one mandatory field: 114

--- PNU: 10,473 records ---
research_id: 0 missing
university: 0 missing
title: 0 missing
publication_year: 0 missing
source: 0 missing
doi: 10,473 missing
url: 10,473 missing
authors: 10,473 missing
Records missing at least one mandatory field: 10,473

Audit only. No files changed.


resume KAUST 2023 and PNU.


In [15]:
# Prepare separate working copies of the additional datasets.
prepared_new_datasets = {
    name: df.copy()
    for name, df in new_datasets.items()
}

print("Working copies ready:", list(prepared_new_datasets))

Working copies ready: ['KAUST_2023', 'PNU']


In [16]:
# Preserve the supplied calendar date; remove only the time component.
kaust_copy = prepared_new_datasets["KAUST_2023"]
date_changes = []
unresolved_dates = []

for index, row in kaust_copy.iterrows():
    original = str(row["publication_date"])
    value = original.strip()

    if value.casefold() in missing_markers:
        continue

    try:
        # Require an explicit year, month and day.
        if not re.fullmatch(
            r"[0-9]{4}-[0-9]{2}-[0-9]{2}(?:[T ].+)?", value
        ):
            raise ValueError

        parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
        standardized = parsed.date().isoformat()

        if original != standardized:
            kaust_copy.at[index, "publication_date"] = standardized
            date_changes.append({
                "research_id": row["research_id"],
                "original_date": original,
                "standardized_date": standardized,
            })

    except ValueError:
        # Keep problematic values unchanged for validation.
        unresolved_dates.append({
            "research_id": row["research_id"],
            "original_date": original,
        })

print("Dates standardized:", len(date_changes))
print("Dates needing review:", len(unresolved_dates))
print("Original input file unchanged.")

Dates standardized: 928
Dates needing review: 0
Original input file unchanged.


All 928 dates were standardized successfully.
Next: combine the validation inputs in memory, including your enriched KSU copy.


In [17]:
# Combine both KAUST files once, then include the other sources.
submission_inputs = {
    "KAUST": pd.concat(
        [
            datasets_to_validate["KAUST"],
            prepared_new_datasets["KAUST_2023"],
        ],
        ignore_index=True,
    ),
    "KFUPM": datasets_to_validate["KFUPM"].copy(),
    "KSU": ksu_enriched.copy(),
    "PNU": prepared_new_datasets["PNU"].copy(),
}

for source, df in submission_inputs.items():
    print(f"{source}: {len(df):,} input records")

print("\nCombined in memory only. Source files unchanged.")

KAUST: 1,042 input records
KFUPM: 48 input records
KSU: 3,535 input records
PNU: 10,473 input records

Combined in memory only. Source files unchanged.


In [18]:
# Find duplicate IDs across all four sources.
all_ids = pd.concat(
    [
        df["research_id"].fillna("").astype(str).str.strip()
        for df in submission_inputs.values()
    ],
    ignore_index=True,
)

present_ids = ~all_ids.str.casefold().isin(missing_markers)
duplicate_ids = set(
    all_ids[present_ids & all_ids.duplicated(keep=False)]
)

# Apply the final rules, including mandatory DOI and authors.
submission_results = {}
submission_summary = []

for source, df in submission_inputs.items():
    result = validate_dataset(
        df,
        expected_university=source,
        duplicate_ids=duplicate_ids,
    )
    submission_results[source] = result

    accepted_count = len(result["accepted"])
    rejected_count = len(result["rejected"])

    submission_summary.append({
        "source": source,
        "input_rows": len(df),
        "accepted_rows": accepted_count,
        "rejected_rows": rejected_count,
    })

    print(f"\n{source}")
    print("Input:", len(df))
    print("Accepted:", accepted_count)
    print("Rejected:", rejected_count)

    # Show the most common combinations of failure reasons.
    if rejected_count:
        print("Failure reasons:")
        print(result["rejected"]["failure_reason"].value_counts().head(5))

print("\nValidation finished in memory. No files overwritten.")


KAUST
Input: 1042
Accepted: 928
Rejected: 114
Failure reasons:
failure_reason
doi: required value missing        113
authors: required value missing      1
Name: count, dtype: Int64

KFUPM
Input: 48
Accepted: 0
Rejected: 48
Failure reasons:
failure_reason
doi: required value missing    48
Name: count, dtype: Int64

KSU
Input: 3535
Accepted: 1477
Rejected: 2058
Failure reasons:
failure_reason
doi: required value missing    2058
Name: count, dtype: Int64

PNU
Input: 10473
Accepted: 0
Rejected: 10473
Failure reasons:
failure_reason
doi: required value missing; url: required value missing; authors: required value missing    10473
Name: count, dtype: Int64

Validation finished in memory. No files overwritten.


In [19]:
total_input = sum(len(df) for df in submission_inputs.values())
total_accepted = sum(
    len(result["accepted"]) for result in submission_results.values()
)
total_rejected = sum(
    len(result["rejected"]) for result in submission_results.values()
)

assert total_input == total_accepted + total_rejected

print("Input:", total_input)
print("Accepted:", total_accepted)
print("Rejected:", total_rejected)
print("All input records accounted for.")

Input: 15098
Accepted: 2405
Rejected: 12693
All input records accounted for.


ORCID may help recover some missing fields, but it won’t necessarily fill every gap.


In [20]:
# get the exact author name for our ORCID trial

# Select the KFUPM thesis we previously inspected.
orcid_sample = submission_inputs["KFUPM"].loc[
    submission_inputs["KFUPM"]["research_id"].eq("KFUPM-142340")
]

assert len(orcid_sample) == 1, "Expected one matching record."

orcid_sample = orcid_sample.iloc[0]

print("Title:", orcid_sample["title"])
print("Authors:", orcid_sample["authors"])
print("University:", orcid_sample["university"])
print("Year:", orcid_sample["publication_year"])


Title: On the Optimal Deployment of Deep Learning Neural Networks on Field Programmable Gate Arrays
Authors: Ahmad Shawahna
University: KFUPM
Year: 2023


In [21]:
import json
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

# Search public researcher profiles by first and family name.
query = 'given-names:Ahmad AND family-name:Shawahna'
search_url = (
    "https://" + "pub.orcid.org/v3.0/search/?"
    + urlencode({"q": query, "rows": 5})
)

request = Request(
    search_url,
    headers={"Accept": "application/json"},
)

orcid_candidates = []

try:
    with urlopen(request, timeout=60) as response:
        orcid_search_bytes = response.read()

    search_result = json.loads(orcid_search_bytes)

    print("Profiles found:", search_result.get("num-found", 0))

    for result in search_result.get("result") or []:
        identifier = result["orcid-identifier"]
        orcid_candidates.append(identifier["path"])
        print("Candidate ORCID:", identifier["path"])

    if not orcid_candidates:
        print("No profiles returned for this name query.")

except HTTPError as error:
    print("HTTP status:", error.code)
    print(error.read().decode("utf-8", errors="replace")[:1000])

except (URLError, TimeoutError) as error:
    print("Connection problem:", error)

print("\nNo research metadata has been changed.")

Profiles found: 2
Candidate ORCID: 0000-0003-3024-8798
Candidate ORCID: 0009-0000-4204-312X

No research metadata has been changed.


In [22]:
# Two profiles match the name. Neither is verified yet. Next, inspect their public names and listed works for the thesis.

import time
from datetime import datetime, timezone

# Preserve the search response and subsequent profile responses.
orcid_dir = project_dir / "data" / "raw" / "orcid"
orcid_dir.mkdir(parents=True, exist_ok=True)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
(orcid_dir / f"search_{stamp}.json").write_bytes(orcid_search_bytes)

orcid_profile_results = {}

for orcid_id in orcid_candidates:
    print(f"\n--- ORCID {orcid_id} ---")

    profile_url = (
        "https://" + f"pub.orcid.org/v3.0/{orcid_id}/record"
    )
    request = Request(
        profile_url,
        headers={"Accept": "application/json"},
    )

    try:
        with urlopen(request, timeout=60) as response:
            profile_bytes = response.read()

        # Save the original API response unchanged.
        (orcid_dir / f"{orcid_id}_{stamp}.json").write_bytes(profile_bytes)
        profile = json.loads(profile_bytes)
        orcid_profile_results[orcid_id] = profile

        name = (profile.get("person") or {}).get("name") or {}
        given = (name.get("given-names") or {}).get("value", "")
        family = (name.get("family-name") or {}).get("value", "")
        print("Public name:", given, family)

        works = (
            (profile.get("activities-summary") or {}).get("works") or {}
        )
        groups = works.get("group") or []
        print("Public work groups:", len(groups))

        # Display potentially relevant titles and their identifiers.
        found_candidate = False

        for group in groups:
            for work in group.get("work-summary") or []:
                title = (
                    ((work.get("title") or {}).get("title") or {})
                    .get("value", "")
                )

                if any(term in title.casefold() for term in [
                    "optimal deployment",
                    "field programmable",
                    "field-programmable",
                    "fpga",
                ]):
                    found_candidate = True
                    print("\nCandidate work:", title)
                    print("Put-code:", work.get("put-code"))
                    print("Type:", work.get("type"))
                    print("Date:", work.get("publication-date"))

                    identifiers = (
                        (work.get("external-ids") or {})
                        .get("external-id") or []
                    )
                    for identifier in identifiers:
                        print(
                            "Identifier:",
                            identifier.get("external-id-type"),
                            identifier.get("external-id-value"),
                            "| Relationship:",
                            identifier.get("external-id-relationship"),
                        )

        if not found_candidate:
            print("No titles matched our trial keywords.")

    except HTTPError as error:
        print("HTTP status:", error.code)
        if error.code in [401, 403, 429]:
            print("Stopping requests; send me this output.")
            break

    except (URLError, TimeoutError) as error:
        print("Connection problem:", error)

    time.sleep(1)

print("\nResearch datasets unchanged.")


--- ORCID 0000-0003-3024-8798 ---
Public name: Ahmad Shawahna
Public work groups: 12

Candidate work: Optimization of FPGA-based CNN Accelerators Using Metaheuristics
Put-code: 127311170
Type: journal-article
Date: {'year': {'value': '2022'}, 'month': None, 'day': None}
Identifier: wosuid PPRN:18419370 | Relationship: self

Candidate work: Optimization of FPGA-based CNN accelerators using metaheuristics
Put-code: 127247147
Type: journal-article
Date: {'year': {'value': '2022'}, 'month': None, 'day': None}
Identifier: doi 10.1007/s11227-022-04787-8 | Relationship: self
Identifier: eid 2-s2.0-85139141289 | Relationship: self
Identifier: issn 15730484 09208542 | Relationship: part-of

Candidate work: FPGA-Based accelerators of deep learning networks for learning and classification: A review
Put-code: 127247150
Type: journal-article
Date: {'year': {'value': '2019'}, 'month': None, 'day': None}
Identifier: doi 10.1109/ACCESS.2018.2890150 | Relationship: self
Identifier: eid 2-s2.0-85060704

ORCID returned related journal articles, but none of the displayed works matches your 2023 thesis. We must not assign those articles’ DOIs to the thesis.
Before concluding this trial, let’s inspect all public work titles from the first profile—our keyword filter may have missed one.

In [23]:
# Inspect every public work title without making another API request.
profile = orcid_profile_results["0000-0003-3024-8798"]

groups = (
    ((profile.get("activities-summary") or {}).get("works") or {})
    .get("group") or []
)

for group in groups:
    for work in group.get("work-summary") or []:
        title = (
            ((work.get("title") or {}).get("title") or {})
            .get("value", "")
        )

        print("\nTitle:", title)
        print("Type:", work.get("type"))
        print("Put-code:", work.get("put-code"))


Title: FxP-QNet: A Post-Training Quantizer for the Design of Mixed Low-Precision DNNs With Dynamic Fixed-Point Representation
Type: journal-article
Put-code: 127191721

Title: FxP-QNet: A Post-Training Quantizer for the Design of Mixed Low-Precision DNNs with Dynamic Fixed-Point Representation
Type: journal-article
Put-code: 127247146

Title: FxP-QNet: A Post-Training Quantizer for the Design of Mixed Low-Precision DNNs With Dynamic Fixed-Point Representation
Type: journal-article
Put-code: 127311145

Title: Optimization of FPGA-based CNN Accelerators Using Metaheuristics
Type: journal-article
Put-code: 127311170

Title: Optimization of FPGA-based CNN accelerators using metaheuristics
Type: journal-article
Put-code: 127247147

Title: EDoS-ADS: An Enhanced Mitigation Technique against Economic Denial of Sustainability (EDoS) Attacks
Type: journal-article
Put-code: 127247149

Title: EDoS-ADS: An Enhanced Mitigation Technique Against Economic Denial of Sustainability (EDoS) Attacks
Type:

None of the listed titles matches the thesis. 
This ORCID trial recovered no DOI for KFUPM-142340. That does 

In [24]:
# not prove the thesis has no DOI elsewhere.
# Let’s record the result, then return to saving your four-source validation outputs.

# Record the limited scope and outcome of this ORCID trial.
orcid_review = {
    "research_id": "KFUPM-142340",
    "title": orcid_sample["title"],
    "profiles_checked": list(orcid_profile_results),
    "profile_identity_status": "Name matches; identity not independently verified",
    "match_status": "no_matching_work_found_in_inspected_public_profiles",
    "confirmed_doi": None,
    "note": (
        "The first profile listed related publications but no matching "
        "thesis title. The second had no public work groups. "
        "DOIs from related articles were not assigned to the thesis."
    ),
    "evidence_directory": orcid_dir.relative_to(project_dir).as_posix(),
}

review_path = (
    project_dir / "data" / "interim"
    / "kfupm_142340_orcid_review.json"
)
review_path.write_text(
    json.dumps(orcid_review, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("ORCID trial documented. No research metadata changed.")

ORCID trial documented. No research metadata changed.


In [25]:
# Create a new folder without replacing previous results.
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
submission_output_dir = (
    project_dir / "data" / "interim"
    / "rana_saad_validation" / f"four_sources_{stamp}"
)
submission_output_dir.mkdir(parents=True, exist_ok=False)

# Combine accepted and rejected records separately.
submission_accepted = pd.concat(
    [
        result["accepted"].drop(columns=["failure_reason"])
        for result in submission_results.values()
    ],
    ignore_index=True,
)

submission_rejected = pd.concat(
    [result["rejected"] for result in submission_results.values()],
    ignore_index=True,
)

assert (
    len(submission_accepted) + len(submission_rejected)
    == sum(len(df) for df in submission_inputs.values())
)
assert submission_accepted["research_id"].str.strip().is_unique

# Save both CSV files and verify that they reopen correctly.
for filename, frame in [
    ("validated.csv", submission_accepted),
    ("rejected.csv", submission_rejected),
]:
    assert not frame["source"].eq("TEST_FIXTURE").any()

    path = submission_output_dir / filename
    frame.to_csv(path, index=False, encoding="utf-8")

    reopened = pd.read_csv(
        path, dtype=str, keep_default_na=False, encoding="utf-8"
    )
    pd.testing.assert_frame_equal(
        reopened, frame.reset_index(drop=True), check_dtype=False
    )

print("Accepted saved:", len(submission_accepted))
print("Rejected preserved:", len(submission_rejected))
print("Saved and verified:", submission_output_dir)

Accepted saved: 2405
Rejected preserved: 12693
Saved and verified: c:\Users\ausu\Desktop\saudi-tech-research\data\interim\rana_saad_validation\four_sources_20260913T112108887260Z


In [26]:
# save the summary and test report in that same folder.


# Save validation counts for all four sources.
pd.DataFrame(submission_summary).to_csv(
    submission_output_dir / "validation_summary.csv",
    index=False,
    encoding="utf-8",
)

# Save the existing validator test results.
assert len(test_results) == 43
assert all(test["status"] == "passed" for test in test_results)

pd.DataFrame(test_results).to_csv(
    submission_output_dir / "validation_test_results.csv",
    index=False,
    encoding="utf-8",
)

# Preserve the KAUST date-format changes for traceability.
pd.DataFrame(
    date_changes,
    columns=["research_id", "original_date", "standardized_date"],
).to_csv(
    submission_output_dir / "kaust_date_changes.csv",
    index=False,
    encoding="utf-8",
)

print("Saved validation summary, 43 test results and KAUST date changes.")

Saved validation summary, 43 test results and KAUST date changes.


In [27]:
# Document the rules and coverage of this four-source run.
four_source_rules = {
    "required_fields": required_fields,
    "optional_fields": optional_fields,
    "missing_markers": sorted(missing_markers),
    "publication_year_range": [min_year, max_year],
    "sources": list(submission_inputs),
    "ksu_year_basis": "KSU annual dataset year",
    "doi_check": "Required; syntax checked, online resolution not guaranteed",
    "enrichment": "Four verified KSU DOIs and three abstracts applied",
    "orcid_trial": "No matching thesis found; no metadata added",
    "accepted_records": len(submission_accepted),
    "rejected_records": len(submission_rejected),
    "status": "Validation outputs; team review and transformation pending",
}

# Save rules and evidence alongside the accepted/rejected CSVs.
for filename, content in [
    ("validation_rules.json", four_source_rules),
    ("enrichment_log.json", enrichment_log),
    ("ksu_provenance.json", ksu_provenance),
    ("orcid_trial_review.json", orcid_review),
]:
    (submission_output_dir / filename).write_text(
        json.dumps(content, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

print("Saved rules, enrichment log, KSU provenance and ORCID review.")

Saved rules, enrichment log, KSU provenance and ORCID review.


In [28]:
# Next: save the rejection counts by rule and source.

from collections import Counter

failure_summary = []

for source, result in submission_results.items():
    # Count each failed rule separately.
    counts = Counter(
        reason
        for reasons in result["rejected"]["failure_reason"]
        for reason in reasons.split("; ")
        if reason
    )

    for reason, count in counts.items():
        failure_summary.append({
            "source": source,
            "failure_reason": reason,
            "failed_rows": count,
        })

pd.DataFrame(
    failure_summary,
    columns=["source", "failure_reason", "failed_rows"],
).to_csv(
    submission_output_dir / "validation_failures_by_rule.csv",
    index=False,
    encoding="utf-8",
)

print("Saved rejection counts by source and rule.")

Saved rejection counts by source and rule.
